# SIC 2026 · Inteligencia Artificial
## Clase 8: Aprendizaje no supervisado (continuación) y taller de segmentación

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

En la Clase 7 vimos precision, recall y F1, y K-medias hasta los ejercicios de escalado. Hoy terminamos el Capítulo 6 y lo aplicamos en un taller completo.

| Hora | Actividad |
|---|---|
| 18:00 a 18:20 | Parte 0. Repaso de K-medias y ejercicios pendientes de la Clase 7 |
| 18:20 a 18:55 | Parte 1. Clustering jerárquico |
| 18:55 a 19:30 | Parte 2. DBSCAN |
| 19:30 a 19:40 | Pausa |
| 19:40 a 20:20 | Parte 3. PCA |
| 20:20 a 21:45 | Parte 4. Taller: segmentación completa de clientes |
| 21:45 a 22:00 | Cierre |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    """Imprime si un ejercicio está bien resuelto."""
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

def cargar_telco():
    try:
        import kagglehub
        carpeta = kagglehub.dataset_download("blastchar/telco-customer-churn")
        return pd.read_csv(os.path.join(carpeta, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))
    except Exception:
        return pd.read_csv("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")

print("Listo.")

---
# Parte 0. Repaso de K-medias

En la Clase 7 segmentamos a los clientes de la compañía de telecomunicaciones usando solo la antigüedad y el cargo mensual:

1. **Escalamos** los datos, porque K-medias usa distancias.
2. Probamos k de 2 a 8 con el **método del codo** y el **coeficiente de silueta**: ambos indicaron k = 4.
3. **Interpretamos** los segmentos y, recién al final, miramos su tasa de fuga: el segmento de clientes nuevos con cargo alto tenía cerca de 50% de fuga, y el de antiguos con cargo bajo, cerca de 5%.

Reconstruimos ese resultado para seguir desde ahí.

In [ ]:
clientes = cargar_telco()
datos_seg = clientes[["tenure", "MonthlyCharges"]]

escalador = StandardScaler().fit(datos_seg)
X_seg = escalador.transform(datos_seg)

km4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_seg)
clientes["segmento"] = km4.labels_

# Centroides en la escala original: inverse_transform deshace el escalado
centroides = escalador.inverse_transform(km4.cluster_centers_)

plt.figure(figsize=(7, 4.5))
sns.scatterplot(data=clientes, x="tenure", y="MonthlyCharges", hue="segmento", palette="tab10", s=8, alpha=0.6)
plt.scatter(centroides[:, 0], centroides[:, 1], s=250, c="red", marker="*", edgecolors="black", label="Centroides")
plt.xlabel("Antigüedad (meses)"); plt.ylabel("Cargo mensual (USD)"); plt.title("Cuatro segmentos y sus centroides")
plt.legend(); plt.show()

clientes.groupby("segmento").agg(
    clientes=("tenure", "size"), antiguedad_media=("tenure", "mean"), cargo_medio=("MonthlyCharges", "mean"),
    tasa_fuga=("Churn", lambda s: (s == "Yes").mean()),
).round(2).sort_values("tasa_fuga", ascending=False)

## Ejercicios pendientes de la Clase 7

### Ejercicio 1
Entrena un K-medias con **k = 3** sobre `X_seg` (con `n_init=10` y `random_state=42`) y guarda su coeficiente de silueta en `silueta_k3`. Usa `sample_size=3000, random_state=0` en `silhouette_score`. ¿Es mejor o peor que el de k = 4 (0,483)?

In [ ]:
silueta_k3 = None  # TU CÓDIGO AQUÍ
print(silueta_k3)

In [ ]:
_ref = silhouette_score(X_seg, KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_seg), sample_size=3000, random_state=0)
revisar(lambda: np.isclose(silueta_k3, _ref))

### Ejercicio 2
¿Qué pasa si **no escalamos**? Imaginemos que el cargo mensual estuviera en **pesos chilenos**. La celda crea `datos_clp` con ese cambio. Entrena un K-medias con k = 4 directamente sobre `datos_clp` (sin escalar), guarda las etiquetas en `etiquetas_sin_escalar` y grafica el resultado. ¿Qué variable domina la división y por qué?

In [ ]:
datos_clp = datos_seg.assign(MonthlyCharges=datos_seg["MonthlyCharges"] * 950)   # tipo de cambio de ejemplo

etiquetas_sin_escalar = None  # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: np.array_equal(etiquetas_sin_escalar, KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(datos_clp)))

---
# Parte 1. Clustering jerárquico

> **Definición. Clustering jerárquico aglomerativo:** parte con cada observación como un grupo propio y, en cada paso, **une los dos grupos más parecidos**, hasta que todo queda en un solo grupo. El resultado es un árbol llamado **dendrograma**.

Ventaja frente a K-medias: no hay que elegir k de antemano. Se mira el dendrograma y se "corta" a la altura que tenga sentido.

> **Criterio de enlace (linkage):** define la distancia entre dos grupos.
>
> - **Ward:** une los grupos que menos aumentan la variación interna. Tiende a formar grupos compactos y de tamaño parecido.
> - **Complete:** usa la distancia entre los dos puntos **más lejanos** de los grupos.
> - **Average:** usa el **promedio** de las distancias entre todos los pares de puntos.
> - **Single:** usa la distancia entre los dos puntos **más cercanos**. Tiende a formar "cadenas".

Usamos el dataset **wine** de scikit-learn: 178 vinos de tres productores, con 13 medidas químicas. Tomamos 40 vinos para que el dendrograma se pueda leer.

In [ ]:
from sklearn.datasets import load_wine
from scipy.cluster.hierarchy import linkage, dendrogram

vinos = load_wine(as_frame=True)
X_vino = StandardScaler().fit_transform(vinos.data)
y_vino = vinos.target          # el productor real: NO se usa para agrupar, solo para comparar al final

rng = np.random.default_rng(0)
idx = rng.choice(len(X_vino), size=40, replace=False)
enlaces = linkage(X_vino[idx], method="ward")

plt.figure(figsize=(12, 4.5))
dendrogram(enlaces, labels=[f"P{p}" for p in y_vino.values[idx]], leaf_font_size=8, color_threshold=12)
plt.axhline(12, ls="--", color="gray")
plt.title("Dendrograma de 40 vinos (la etiqueta indica el productor real)")
plt.ylabel("Distancia de unión"); plt.show()

**Cómo leerlo:** abajo cada vino es un grupo. Hacia arriba se van uniendo. La altura de cada unión indica cuán distintos eran los grupos que se unieron. Si cortamos con la línea gris, quedan **3 grupos**, y cada grupo concentra mayoritariamente a un productor, aunque el algoritmo nunca vio esa información. Hay algunos vinos mezclados: el agrupamiento no es perfecto, y eso es normal.

**¿Dónde cortar?** Donde haya un "salto" grande de altura entre uniones consecutivas: significa que se estarían uniendo grupos muy distintos.

Con scikit-learn, para todo el dataset:

In [ ]:
from sklearn.cluster import AgglomerativeClustering

jerarquico = AgglomerativeClustering(n_clusters=3, linkage="ward").fit(X_vino)
pd.crosstab(jerarquico.labels_, y_vino, rownames=["Grupo encontrado"], colnames=["Productor real"])

Los números de los grupos son arbitrarios: el "grupo 0" no tiene por qué ser el "productor 0". Lo que importa es que cada grupo concentre a un productor.

### ¿Importa el criterio de enlace?

Para comparar, usamos el **índice de Rand ajustado (ARI)**, que mide cuánto coincide una agrupación con otra: 1 es coincidencia perfecta y 0 es lo mismo que el azar. Solo se puede usar cuando, excepcionalmente, conocemos la respuesta real, como aquí.

In [ ]:
from sklearn.metrics import adjusted_rand_score

for criterio in ["ward", "complete", "average", "single"]:
    etiquetas = AgglomerativeClustering(n_clusters=3, linkage=criterio).fit_predict(X_vino)
    tamanos = np.bincount(etiquetas)
    print(f"{criterio:9s} ARI = {adjusted_rand_score(y_vino, etiquetas):.3f} | tamaño de los grupos: {tamanos}")

El criterio de enlace cambia todo. Con **single** y **average**, casi todos los vinos quedan en un solo grupo gigante y los otros dos grupos tienen uno o dos vinos: el efecto "cadena". **Ward** es la opción por defecto razonable.

### Ejercicio 3
Repite el dendrograma de los 40 vinos usando `method="single"` y guarda el resultado de `linkage` en `enlaces_single`. Grafícalo y compáralo con el de Ward: ¿se ven grupos claros?

In [ ]:
enlaces_single = None  # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: np.allclose(enlaces_single, linkage(X_vino[idx], method="single")))

---
# Parte 2. DBSCAN: grupos con forma libre

K-medias supone que los grupos son "bolitas" alrededor de un centroide. Cuando los grupos tienen otras formas, falla.

> **Definición. DBSCAN:** forma grupos con las zonas donde hay **muchos puntos juntos** (alta densidad). Dos parámetros:
>
> - `eps`: el radio del vecindario de cada punto.
> - `min_samples`: cuántos vecinos necesita un punto dentro de ese radio para ser "núcleo" de un grupo.
>
> Los puntos que no quedan en ninguna zona densa se marcan como **ruido** (etiqueta -1). No hay que elegir k.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

X_luna, y_luna = make_moons(n_samples=300, noise=0.07, random_state=0)
X_luna = StandardScaler().fit_transform(X_luna)

etiquetas_km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X_luna)
etiquetas_db = DBSCAN(eps=0.3, min_samples=5).fit_predict(X_luna)

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
ejes[0].scatter(X_luna[:, 0], X_luna[:, 1], c=etiquetas_km, cmap="coolwarm", s=15); ejes[0].set_title("K-medias (k = 2)")
ejes[1].scatter(X_luna[:, 0], X_luna[:, 1], c=etiquetas_db, cmap="coolwarm", s=15); ejes[1].set_title("DBSCAN (eps = 0,3)")
plt.show()
print(f"ARI K-medias: {adjusted_rand_score(y_luna, etiquetas_km):.3f} | ARI DBSCAN: {adjusted_rand_score(y_luna, etiquetas_db):.3f}")
print("Grupos DBSCAN:", sorted(set(etiquetas_db)), "| Puntos de ruido:", (etiquetas_db == -1).sum())

K-medias corta las lunas por la mitad con una línea recta. DBSCAN sigue la forma de cada luna.

### ¿Cómo elegir eps? El gráfico de k-distancia

Elegir `eps` a ojo es difícil. Un método práctico:

1. Para cada punto, calcular la distancia a su vecino número `min_samples`.
2. Ordenar esas distancias de menor a mayor y graficarlas.
3. Buscar el **codo**: donde la curva empieza a subir rápido. Los puntos a la derecha del codo están lejos de todo (son ruido), y el valor del codo es un buen `eps`.

In [ ]:
from sklearn.neighbors import NearestNeighbors

vecinos = NearestNeighbors(n_neighbors=5).fit(X_luna)
distancias, _ = vecinos.kneighbors(X_luna)
k_distancia = np.sort(distancias[:, -1])     # distancia al 5.º vecino (el primero es el propio punto)

plt.figure(figsize=(7, 4))
plt.plot(k_distancia)
plt.axhline(0.3, ls="--", color="gray", label="eps = 0,3")
plt.xlabel("Puntos ordenados por distancia"); plt.ylabel("Distancia al 5.º vecino")
plt.title("Gráfico de k-distancia: el codo sugiere eps"); plt.legend(); plt.show()

### DBSCAN para detectar anomalías

Como DBSCAN marca como ruido a los puntos aislados, sirve para **detectar casos raros**: transacciones sospechosas, sensores con lecturas anómalas o errores de digitación. Agregamos 10 puntos "extraños" a las lunas y vemos si los detecta.

In [ ]:
rng = np.random.default_rng(1)
anomalias = rng.uniform(-2.5, 2.5, size=(10, 2))
X_con_anomalias = np.vstack([X_luna, anomalias])

etiquetas = DBSCAN(eps=0.3, min_samples=5).fit_predict(X_con_anomalias)
es_anomalia = np.r_[np.zeros(len(X_luna)), np.ones(10)].astype(bool)

plt.figure(figsize=(7, 4.5))
plt.scatter(X_con_anomalias[:, 0], X_con_anomalias[:, 1], c=(etiquetas == -1), cmap="coolwarm", s=18)
plt.scatter(anomalias[:, 0], anomalias[:, 1], facecolors="none", edgecolors="black", s=120, label="Anomalías agregadas")
plt.title("En rojo, los puntos que DBSCAN marcó como ruido"); plt.legend(); plt.show()
print("Anomalías agregadas detectadas como ruido:", int((etiquetas[es_anomalia] == -1).sum()), "de 10")

Algunas anomalías quedan cerca de las lunas y DBSCAN las considera parte de un grupo. Es una limitación esperable: un punto raro que cae dentro de una zona densa no se distingue de los demás.

### Ejercicio 4
Prueba DBSCAN sobre `X_luna` con `eps=0.1`, `eps=0.2` y `eps=0.5` (con `min_samples=5`). Para cada uno, guarda en el diccionario `n_grupos` cuántos grupos encuentra **sin contar el ruido**. ¿Qué pasa con un eps muy chico y con uno muy grande? Compáralo con el gráfico de k-distancia.

In [ ]:
n_grupos = {}
for eps in [0.1, 0.2, 0.5]:
    pass  # TU CÓDIGO AQUÍ
n_grupos

In [ ]:
_esp = {e: len(set(DBSCAN(eps=e, min_samples=5).fit_predict(X_luna)) - {-1}) for e in [0.1, 0.2, 0.5]}
revisar(lambda: n_grupos == _esp)

---
# Parte 3. PCA: resumir muchas columnas en pocas

> **Definición. Análisis de componentes principales (PCA):** crea nuevas variables, llamadas **componentes**, que son combinaciones de las originales. La primera componente es la dirección en que los datos **varían más**; la segunda, la que varía más entre las perpendiculares a la primera, y así sucesivamente.

Para qué sirve:

1. **Visualizar** datos de muchas dimensiones en 2D.
2. **Reducir** el número de columnas antes de un modelo, cuando hay muchas y están correlacionadas.
3. **Quitar ruido**, quedándose con las componentes que explican más variación.

> **Varianza explicada:** proporción de la variación total de los datos que captura cada componente.

Los vinos tienen 13 medidas químicas: imposible de graficar. Con PCA las resumimos en 2.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(X_vino)
var = pca.explained_variance_ratio_

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
ejes[0].bar(range(1, 14), var, color="#1428A0", label="Por componente")
ejes[0].plot(range(1, 14), var.cumsum(), marker="o", color="orange", label="Acumulada")
ejes[0].set_xlabel("Componente"); ejes[0].set_ylabel("Varianza explicada"); ejes[0].set_title("¿Cuánta información guarda cada componente?")
ejes[0].legend()

pca2 = PCA(n_components=2).fit(X_vino)
Z = pca2.transform(X_vino)
sc = ejes[1].scatter(Z[:, 0], Z[:, 1], c=y_vino, cmap="viridis", s=20)
ejes[1].set_xlabel("Componente 1"); ejes[1].set_ylabel("Componente 2"); ejes[1].set_title("13 medidas resumidas en 2 (color: productor real)")
plt.colorbar(sc, ax=ejes[1]); plt.tight_layout(); plt.show()

print(f"Las 2 primeras componentes explican el {var[:2].sum():.1%} de la variación total")

Con solo 2 de 13 dimensiones, que guardan algo más de la mitad de la variación, los tres productores se ven claramente separados.

### ¿Qué significa cada componente? Las cargas

Cada componente es una combinación de las variables originales. Los **coeficientes** de esa combinación se llaman **cargas** (loadings): indican cuánto pesa cada variable. Mirar las cargas más grandes (en valor absoluto) permite ponerle nombre a una componente.

In [ ]:
cargas = pd.DataFrame(pca2.components_.T, index=vinos.data.columns, columns=["Componente 1", "Componente 2"])

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
for eje, col in zip(ejes, cargas.columns):
    cargas[col].sort_values().plot(kind="barh", ax=eje, color="#1428A0", title=f"Cargas de la {col.lower()}")
plt.tight_layout(); plt.show()

La componente 1 combina sobre todo fenoles, flavonoides y la medida de dilución (od280), con signo positivo, y el ácido málico y otras con signo negativo: resume la "intensidad fenólica" del vino. La componente 2 está dominada por el color, el alcohol y la prolina: resume el "cuerpo" del vino. No siempre se puede nombrar una componente con tanta claridad, pero vale la pena intentarlo. Ojo: el signo de una componente es arbitrario y puede aparecer invertido; lo que importa es qué variables van juntas y cuáles en sentido contrario.

### PCA + K-medias

In [ ]:
km_13 = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_vino)
km_2 = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(Z)
print(f"K-medias con las 13 medidas: ARI = {adjusted_rand_score(y_vino, km_13):.3f}")
print(f"K-medias con 2 componentes:   ARI = {adjusted_rand_score(y_vino, km_2):.3f}")

Con 2 componentes se obtiene prácticamente el mismo resultado que con las 13 columnas: PCA comprimió los datos sin perder lo importante para separar los grupos.

### PCA antes de un modelo supervisado

El dataset **digits** tiene imágenes de dígitos escritos a mano de 8 × 8 pixeles: 64 columnas. ¿Cuánto perdemos si las resumimos con PCA antes de clasificar? Ponemos PCA **dentro del Pipeline**, para que se ajuste solo con los datos de entrenamiento de cada pliegue.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

digitos = load_digits()
fig, ejes = plt.subplots(1, 8, figsize=(10, 1.6))
for eje, img, etiqueta in zip(ejes, digitos.images, digitos.target):
    eje.imshow(img, cmap="gray_r"); eje.set_title(etiqueta); eje.axis("off")
plt.show()

resultados = []
for n in [5, 10, 20, 30, 64]:
    pipe = Pipeline([("escalar", StandardScaler()), ("pca", PCA(n_components=n)), ("modelo", LogisticRegression(max_iter=3000))])
    resultados.append({"Componentes": n, "Exactitud CV": cross_val_score(pipe, digitos.data, digitos.target, cv=5).mean()})
pd.DataFrame(resultados).round(3)

Con 30 componentes, menos de la mitad de las 64 columnas, la exactitud baja muy poco. En datasets grandes, esto hace los modelos más rápidos y livianos. Con muy pocas componentes, en cambio, se pierde información útil y la exactitud cae.

### Ejercicio 5
¿Cuántas componentes se necesitan para explicar **al menos el 80%** de la variación de los vinos? Usa `var.cumsum()` y guarda la respuesta en `n_componentes_80`.

In [ ]:
n_componentes_80 = None  # TU CÓDIGO AQUÍ
print(n_componentes_80)

In [ ]:
revisar(lambda: n_componentes_80 == int(np.argmax(var.cumsum() >= 0.8) + 1))

---
# Parte 4. Taller: segmentación completa de clientes

En la Clase 7 segmentamos con solo 2 variables. Una compañía real usaría toda la información: qué servicios tiene cada cliente, su contrato, su forma de pago. Ahora hacemos la segmentación completa, aplicando todo el capítulo.

**Objetivo del taller:** encontrar segmentos de clientes que el equipo comercial pueda entender y usar, y responder si esos segmentos ayudan a predecir la fuga.

## Paso 1. Preparar los datos

Usamos variables numéricas (antigüedad y cargos) y los servicios contratados. Igual que en la Clase 6, `TotalCharges` viene como texto con 11 vacíos de clientes nuevos: los convertimos a número y los rellenamos con 0, porque esos clientes aún no han pagado nada.

**No incluimos `Churn`:** es lo que queremos explicar al final, y el clustering no debe verlo.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

clientes["TotalCharges"] = pd.to_numeric(clientes["TotalCharges"], errors="coerce").fillna(0)

NUM = ["tenure", "MonthlyCharges", "TotalCharges"]
CAT = ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity", "TechSupport", "StreamingTV",
       "StreamingMovies", "PaperlessBilling", "PhoneService", "MultipleLines", "OnlineBackup", "DeviceProtection"]

prep_seg = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("cat", OneHotEncoder(sparse_output=False), CAT),
])
X_full = prep_seg.fit_transform(clientes)
print("Clientes:", X_full.shape[0], "| Columnas después de transformar:", X_full.shape[1])

## Paso 2. Visualizar con PCA

38 columnas no se pueden graficar. Con PCA las bajamos a 2 solo para **mirar** los datos: el clustering lo haremos con todas las columnas.

In [ ]:
pca_full = PCA(n_components=2).fit(X_full)
Z_full = pca_full.transform(X_full)
print(f"Varianza explicada por 2 componentes: {pca_full.explained_variance_ratio_.sum():.1%}")

plt.figure(figsize=(7, 4.5))
plt.scatter(Z_full[:, 0], Z_full[:, 1], s=4, alpha=0.3)
plt.xlabel("Componente 1"); plt.ylabel("Componente 2"); plt.title("Clientes en 2D con PCA")
plt.show()

Se ven agrupaciones: ya hay estructura que un clustering puede encontrar.

## Paso 3. Elegir k

### Ejercicio 6
Para k de 2 a 7, entrena un K-medias sobre `X_full` (con `n_init=10` y `random_state=42`) y calcula el coeficiente de silueta (con `sample_size=3000, random_state=0`). Guarda los resultados en el diccionario `siluetas_full`, con k como clave. ¿Qué k recomienda la silueta?

In [ ]:
siluetas_full = {}
for k in range(2, 8):
    pass  # TU CÓDIGO AQUÍ
siluetas_full

In [ ]:
_k = 5
_r = silhouette_score(X_full, KMeans(n_clusters=_k, n_init=10, random_state=42).fit_predict(X_full), sample_size=3000, random_state=0)
revisar(lambda: len(siluetas_full) == 6 and np.isclose(siluetas_full[_k], _r))

**Dos observaciones importantes:**

1. Los valores de silueta son **más bajos** que en la Clase 7 (alrededor de 0,3 contra 0,48). Es normal: con muchas columnas, sobre todo categóricas, los grupos están menos separados. No significa que la segmentación sea mala.
2. La silueta favorece k = 3, pero **la silueta no es la única guía**. Comparamos k = 3 y k = 4 por lo que más importa: si los segmentos sirven para el negocio.

## Paso 4. Interpretar los segmentos

In [ ]:
def perfil_segmentos(k):
    etiquetas = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_full)
    tabla = clientes.assign(seg=etiquetas).groupby("seg").agg(
        clientes=("tenure", "size"),
        antiguedad=("tenure", "mean"),
        cargo_mensual=("MonthlyCharges", "mean"),
        sin_internet=("InternetService", lambda s: (s == "No").mean()),
        fibra=("InternetService", lambda s: (s == "Fiber optic").mean()),
        mes_a_mes=("Contract", lambda s: (s == "Month-to-month").mean()),
        streaming_tv=("StreamingTV", lambda s: (s == "Yes").mean()),
        tasa_fuga=("Churn", lambda s: (s == "Yes").mean()),
    ).round(2)
    return etiquetas, tabla.sort_values("tasa_fuga", ascending=False)

etiquetas_k3, perfil_k3 = perfil_segmentos(3)
etiquetas_k4, perfil_k4 = perfil_segmentos(4)
print("k = 3"); display(perfil_k3)
print("k = 4"); display(perfil_k4)

**Con k = 3** salen tres segmentos claros: clientes sin internet (fuga muy baja), clientes nuevos con internet y contrato mes a mes (fuga alta) y clientes antiguos con muchos servicios (fuga baja).

**Con k = 4**, el segmento de fuga alta se divide en dos: clientes con **fibra óptica**, contrato mes a mes y cargo alto, con una fuga cercana al 57%, y clientes con **DSL** y cargo medio, con una fuga cercana al 25%.

Esa división es muy útil para el negocio: dice que el problema de fuga se concentra en los clientes nuevos de fibra. Por eso, aunque la silueta prefiere k = 3, **k = 4 es mejor elección para este objetivo**. Elegir k es una decisión técnica **y** de negocio.

### Ejercicio 7
Ponle un **nombre de negocio** a cada uno de los 4 segmentos de `perfil_k4` (por ejemplo, "Fieles sin internet"). Guarda un diccionario `nombres_segmentos` con el número de segmento como clave y el nombre como valor. Este ejercicio no tiene verificación automática: discútanlo en grupo.

In [ ]:
nombres_segmentos = {}  # TU CÓDIGO AQUÍ, por ejemplo {0: "...", 1: "...", 2: "...", 3: "..."}

## Paso 5. Ver los segmentos en el mapa de PCA

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))
for eje, etiquetas, titulo in [(ejes[0], etiquetas_k4, "Segmentos (k = 4)"),
                               (ejes[1], (clientes["Churn"] == "Yes").astype(int), "Fuga real (1 = se fue)")]:
    sc = eje.scatter(Z_full[:, 0], Z_full[:, 1], c=etiquetas, cmap="tab10" if "Segmentos" in titulo else "coolwarm", s=4, alpha=0.5)
    eje.set_title(titulo); eje.set_xlabel("Componente 1"); eje.set_ylabel("Componente 2")
    plt.colorbar(sc, ax=eje)
plt.tight_layout(); plt.show()

A la izquierda los segmentos; a la derecha, la fuga real. Las zonas con más fuga coinciden con uno de los segmentos, aunque el clustering nunca vio esa columna.

## Paso 6. ¿Los segmentos mejoran el modelo de fuga?

Una idea común: usar el segmento como característica extra en el modelo supervisado. Lo probamos sin fuga de datos: el K-medias va **dentro del Pipeline**, así que en cada pliegue de la validación cruzada se ajusta solo con los datos de entrenamiento.

`FeatureUnion` junta dos transformaciones en paralelo: las columnas originales (`passthrough`) y la **distancia de cada cliente a cada centroide** (`KMeans.transform`).

In [ ]:
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import StratifiedKFold

y_fuga = (clientes["Churn"] == "Yes").astype(int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelo_base = Pipeline([("prep", prep_seg), ("modelo", LogisticRegression(max_iter=3000))])
modelo_con_segmentos = Pipeline([
    ("prep", prep_seg),
    ("union", FeatureUnion([("originales", "passthrough"),
                            ("distancias", KMeans(n_clusters=4, n_init=10, random_state=42))])),
    ("modelo", LogisticRegression(max_iter=3000)),
])

auc_base = cross_val_score(modelo_base, clientes, y_fuga, cv=cv, scoring="roc_auc").mean()
auc_seg = cross_val_score(modelo_con_segmentos, clientes, y_fuga, cv=cv, scoring="roc_auc").mean()
print(f"ROC AUC sin segmentos: {auc_base:.4f}")
print(f"ROC AUC con segmentos: {auc_seg:.4f}")

**La mejora es mínima.** ¿Por qué? Porque los segmentos se construyeron con las mismas variables que el modelo ya tiene: no le entregan información nueva, solo la reorganizan.

Esta es una conclusión valiosa, no un fracaso. El clustering no siempre mejora la predicción, pero sí sirve para **entender** a los clientes y **comunicar** el problema: "la fuga se concentra en los clientes nuevos de fibra con contrato mes a mes" es una frase que un gerente puede usar para decidir.

---
# Cierre

| Método | Qué hace | Hay que elegir | Fortaleza | Debilidad |
|---|---|---|---|---|
| K-medias | Grupos alrededor de centroides | k | Rápido y fácil de interpretar | Supone grupos redondos y de tamaño parecido |
| Jerárquico | Une grupos paso a paso | Criterio de enlace y dónde cortar | Muestra la estructura completa | Lento con muchos datos; sensible al enlace |
| DBSCAN | Grupos por densidad | eps y min_samples | Formas libres, detecta ruido y anomalías | Sensible a eps; difícil con densidades distintas |
| PCA | Resume columnas en componentes | Número de componentes | Visualizar y comprimir | Las componentes pueden ser difíciles de interpretar |

**Ideas para llevar al capstone:**

- Usen PCA en el EDA para visualizar datos con muchas columnas.
- Si su proyecto tiene clientes, usuarios o productos, una segmentación puede ser una sección valiosa del informe.
- Elegir parámetros (k, eps, componentes) combina métricas y criterio de negocio.

Guarden el notebook en `ejercicios_estudiantes` como `clase08_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1

In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_seg)
silueta_k3 = silhouette_score(X_seg, km3.labels_, sample_size=3000, random_state=0)
print(round(silueta_k3, 3), "(peor que k = 4)")

### Ejercicio 2
En pesos, el cargo mensual va de unos 17.000 a 113.000, mientras la antigüedad va de 0 a 72. En la distancia, una diferencia de 1.000 pesos pesa lo mismo que 1.000 meses, así que el cargo domina todo: los grupos quedan como franjas horizontales y la antigüedad no influye. La misma información, con otra unidad, da un clustering completamente distinto. Por eso hay que escalar.

In [ ]:
etiquetas_sin_escalar = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(datos_clp)
plt.figure(figsize=(7, 4.5))
plt.scatter(datos_clp["tenure"], datos_clp["MonthlyCharges"], c=etiquetas_sin_escalar, cmap="tab10", s=5)
plt.xlabel("Antigüedad (meses)"); plt.ylabel("Cargo mensual (CLP)"); plt.title("K-medias sin escalar: domina el cargo en pesos")
plt.show()

### Ejercicio 3
Con `single` no se ven grupos claros: los vinos se van uniendo uno a uno a un grupo que crece como una cadena.

In [ ]:
enlaces_single = linkage(X_vino[idx], method="single")
plt.figure(figsize=(12, 4.5))
dendrogram(enlaces_single, labels=[f"P{p}" for p in y_vino.values[idx]], leaf_font_size=8)
plt.title("Dendrograma con enlace single: efecto cadena"); plt.show()

### Ejercicio 4
Con eps muy chico, casi todos los puntos quedan aislados: muchos grupos pequeños y mucho ruido. Con eps muy grande, todo queda en un solo grupo. El codo del gráfico de k-distancia (cerca de 0,3) indica el rango razonable.

In [ ]:
n_grupos = {}
for eps in [0.1, 0.2, 0.5]:
    etiquetas = DBSCAN(eps=eps, min_samples=5).fit_predict(X_luna)
    n_grupos[eps] = len(set(etiquetas) - {-1})
n_grupos

### Ejercicio 5

In [ ]:
n_componentes_80 = int(np.argmax(var.cumsum() >= 0.8) + 1)   # argmax devuelve la primera posición que cumple
print(n_componentes_80, "componentes explican", round(var.cumsum()[n_componentes_80 - 1], 3))

### Ejercicio 6

In [ ]:
siluetas_full = {}
for k in range(2, 8):
    etiquetas = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_full)
    siluetas_full[k] = silhouette_score(X_full, etiquetas, sample_size=3000, random_state=0)
print({k: round(v, 3) for k, v in siluetas_full.items()}, "| mejor k según silueta:", max(siluetas_full, key=siluetas_full.get))

### Ejercicio 7
Una propuesta (los números de segmento pueden variar según la versión de scikit-learn; guíense por el perfil):

- **Nuevos de fibra, mes a mes:** cargo alto, poca antigüedad, fuga cercana al 57%. Prioridad de retención.
- **Internet DSL, cargo medio:** fuga cercana al 25%.
- **Antiguos con todo incluido:** muchos servicios, contratos largos, fuga cercana al 14%.
- **Fieles sin internet:** solo telefonía, cargo bajo, fuga cercana al 7%.